# Exploración del Dataset NoisyUAV v2

**Objetivo:** Visualización y comprensión de las señales RF I/Q del dataset para detección de drones.

**Dataset:** Glüge et al. (2024) — *Robust Low-Cost Drone Detection and Classification Using CNNs in Low SNR Environments*

| Parámetro | Valor |
|---|---|
| Fs (muestreo) | 14 MHz (downsampled de 56 MHz) |
| Muestras/archivo | 1,048,576 (2²⁰) |
| Duración | ~74.9 ms |
| Clases | 6 drones + 1 ruido = 7 |
| SNR | [-20, 30] dB, paso 2 dB |
| Total muestras | 17,744 |

In [ ]:
import sys
import os

# Añadir la carpeta padre al path para poder importar 'funciones'
sys.path.insert(0, r"c:\repos\DroneDetectionRF")

from NoisyUAV.funciones.dataset.cargador import (
    cargar_muestra,
    cargar_metadatos_dataset,
    obtener_una_muestra,
    obtener_muestras_por_clase,
    NOMBRES_CLASES,
    FREQ_MUESTREO,
    DURACION_MUESTRA,
    DATA_DIR,
    TARGET_NOISE,
)
from NoisyUAV.funciones.visualizacion.visualizacion import (
    panel_completo,
    comparar_snr,
    comparar_clases,
)
# from NoisyUAV.funciones.cargador import obtener_una_muestra

import matplotlib.pyplot as plt
%matplotlib inline

print(f"Directorio de datos: {DATA_DIR}")
print(f"Frecuencia de muestreo: {FREQ_MUESTREO/1e6} MHz")
print(f"Duración por muestra: {DURACION_MUESTRA*1000:.1f} ms")
print(f"Clases: {NOMBRES_CLASES}")
print(f"Índice de clase Noise: {TARGET_NOISE}")

## 2. Metadatos del Dataset

In [ ]:
class_stats, snr_stats = cargar_metadatos_dataset()

print("=" * 50)
print("DISTRIBUCIÓN POR CLASE")
print("=" * 50)
display(class_stats)

print("\n")
print("=" * 50)
print("DISTRIBUCIÓN POR SNR")
print("=" * 50)
display(snr_stats)

## 3. Cargar y Examinar una Muestra

In [ ]:
# Elegimos un dron Taranis (target=5) a SNR alto para ver la señal limpia
rutas = obtener_muestras_por_clase(target=0, snr=-20, n=10)
rutas

In [ ]:
iq, sample_id, target, snr = cargar_muestra(rutas[0])

print(f"Archivo: {os.path.basename(rutas[0])}")
print(f"Shape del tensor: {iq.shape}")
print(f"Clase (target): {target} → {NOMBRES_CLASES[target]}")
print(f"SNR: {snr} dB")
print(f"Sample ID: {sample_id}")
print(f"Rango I: [{iq[0].min():.6f}, {iq[0].max():.6f}]")
print(f"Rango Q: [{iq[1].min():.6f}, {iq[1].max():.6f}]")
print(f"Tipo de dato: {iq.dtype}")

## 4. Panel Completo de Visualización

Visualización en 4 dominios de una señal de dron a **alto SNR** (señal limpia, fácil de interpretar).

In [ ]:
resultado_dron = obtener_una_muestra(target=4, snr=-16, index=6)
tensor_dron = resultado_dron[0]

In [ ]:
resultado_dron

In [ ]:
fig = panel_completo(tensor_dron, 4, -16)
plt.show()

In [ ]:
iq_noise, sid_noise, tgt_noise, snr_noise = obtener_una_muestra(target=TARGET_NOISE, snr=-10)
fig = panel_completo(iq_noise, tgt_noise, snr_noise, sample_id=sid_noise)
plt.show()

## 5. Comparación por SNR

¿Cómo se degrada la señal de un mismo dron a medida que baja el SNR?

Esto es clave para entender el **reto de detección a SNR negativos**.

In [ ]:
fig = comparar_snr(
    target=2,  # Taranis (índice 5 en orden alfabético)
    snr_list=[-20, -10, 0, 10, 20, 30],
)
plt.show()

## 6. Comparación por Clase

¿Cómo se ven las distintas firmas RF de cada dron al mismo SNR?

Cada dron tiene un patrón de **frequency hopping** y una **modulación** característica.

In [ ]:
fig = comparar_clases(
    snr=-12,
    targets=[0, 1, 2, 3, 4, 5, 6],  # Todas las clases (4=Noise)
)
plt.show()